# Summary Table viewer

Loads the per-asset stats JSON files produced by `scripts/dor_run.py` and
renders the 22 × 8 Summary Table without re-running any computation.

Use this notebook to inspect the final report after a run, before opening the
Excel xlsx. To regenerate: `python scripts/dor_run.py --allow-empty`.

In [1]:
import csv
import json
import re
import sys
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))

CONFIG = ROOT / 'config' / 'assets.csv'
OUTPUT = ROOT / 'output'

def latest_stats(ticker, tf):
    pattern = re.compile(rf'^{re.escape(ticker)}_{tf}_(\d{{4}}-\d{{2}}-\d{{2}})_stats\.json$')
    candidates = []
    for p in OUTPUT.iterdir():
        m = pattern.match(p.name)
        if m:
            candidates.append((m.group(1), p))
    if not candidates:
        return None
    candidates.sort()
    return json.loads(candidates[-1][1].read_text())

with CONFIG.open() as f:
    assets = list(csv.DictReader(f))
print(f'{len(assets)} assets in config')

22 assets in config


In [2]:
rows = []
for asset in assets:
    ticker = asset['ticker']
    row = {'Ticker': ticker, 'Asset Class': asset['asset_class']}
    for tf, label in [('d', 'Daily'), ('w', 'Weekly'), ('m', 'Monthly'), ('q', 'Quarterly')]:
        payload = latest_stats(ticker, tf)
        if payload is None:
            row[f'C-C StdDev {label}'] = None
            row[f'H-L Mean {label}']   = None
            continue
        cc = payload['returns'].get('C-C Returns', {}).get('stats', {})
        hl = payload['returns'].get('H-L Returns', {}).get('stats', {})
        row[f'C-C StdDev {label}'] = cc.get('standard_deviation')
        row[f'H-L Mean {label}']   = hl.get('mean')
    rows.append(row)

summary = pd.DataFrame(rows).set_index('Ticker')
summary

,Asset Class,C-C StdDev Daily,H-L Mean Daily,C-C StdDev Weekly,H-L Mean Weekly,C-C StdDev Monthly,H-L Mean Monthly,C-C StdDev Quarterly,H-L Mean Quarterly
Ticker,,,,,,,,,
EURUSD,FX Major,0.006969,0.007414,0.012522,0.019937,0.026056,0.044828,0.045373,0.082565
USDJPY,FX Major,0.007383,0.008591,0.014897,0.021884,0.029360,0.047691,0.052881,0.083620
GBPUSD,FX Major,0.005893,0.007849,0.013406,0.020380,0.024499,0.046145,0.045256,0.089172
AUDUSD,FX Major,0.007882,0.009754,0.016818,0.024658,0.035573,0.054459,0.063389,0.098745
USDZAR,FX EM,0.011986,0.016271,0.029810,0.044091,0.046302,0.103283,0.077300,0.211044
USDBRL,FX EM,0.011543,0.015319,0.023274,0.035820,0.048104,0.072672,0.084655,0.133273
USDTRY,FX EM,0.010109,0.011207,0.024430,0.028861,0.051890,0.066227,0.094560,0.131582
SP500,Equity Index,0.013507,0.009170,0.028433,0.029349,0.043673,0.071716,0.067984,0.130542
NDX100,Equity Index,0.016298,0.017951,0.032701,0.046450,0.067687,0.108162,0.113508,0.204029


In [3]:
# Format as percentages.
metric_cols = [c for c in summary.columns if c != 'Asset Class']
summary.style.format({c: '{:.2%}' for c in metric_cols}, na_rep='-')

,Asset Class,C-C StdDev Daily,H-L Mean Daily,C-C StdDev Weekly,H-L Mean Weekly,C-C StdDev Monthly,H-L Mean Monthly,C-C StdDev Quarterly,H-L Mean Quarterly
Ticker,,,,,,,,,
EURUSD,FX Major,0.70%,0.74%,1.25%,1.99%,2.61%,4.48%,4.54%,8.26%
USDJPY,FX Major,0.74%,0.86%,1.49%,2.19%,2.94%,4.77%,5.29%,8.36%
GBPUSD,FX Major,0.59%,0.78%,1.34%,2.04%,2.45%,4.61%,4.53%,8.92%
AUDUSD,FX Major,0.79%,0.98%,1.68%,2.47%,3.56%,5.45%,6.34%,9.87%
USDZAR,FX EM,1.20%,1.63%,2.98%,4.41%,4.63%,10.33%,7.73%,21.10%
USDBRL,FX EM,1.15%,1.53%,2.33%,3.58%,4.81%,7.27%,8.47%,13.33%
USDTRY,FX EM,1.01%,1.12%,2.44%,2.89%,5.19%,6.62%,9.46%,13.16%
SP500,Equity Index,1.35%,0.92%,2.84%,2.93%,4.37%,7.17%,6.80%,13.05%
NDX100,Equity Index,1.63%,1.80%,3.27%,4.64%,6.77%,10.82%,11.35%,20.40%
